Dataset loading

In [ ]:

from matminer.datasets import load_dataset
import pandas as pd
df=load_dataset('matbench_steels')
print(df.head())
print('\nDataset Shape:',df.shape)
print(df.columns)

                                         composition  yield strength
0  Fe0.620C0.000953Mn0.000521Si0.00102Cr0.000110N...          2411.5
1  Fe0.623C0.00854Mn0.000104Si0.000203Cr0.147Ni0....          1123.1
2  Fe0.625Mn0.000102Si0.000200Cr0.0936Ni0.129Mo0....          1736.3
3  Fe0.634C0.000478Mn0.000523Si0.00102Cr0.000111N...          2487.3
4  Fe0.636C0.000474Mn0.000518Si0.00101Cr0.000109N...          2249.6

Dataset Shape: (312, 2)
Index(['composition', 'yield strength'], dtype='object')


Composition featurization


In [ ]:
from pymatgen.core import Composition

# Convert string formulas to pymatgen Composition objects
df["composition"] = df["composition"].apply(Composition)

from matminer.featurizers.composition import ElementProperty
ep_featurizer=ElementProperty.from_preset('magpie')
df_features = ep_featurizer.featurize_dataframe(df, col_id="composition")
print(df_features.head())
print("\nNew Feature Dataset Shape:", df_features.shape)

ElementProperty:   0%|          | 0/312 [00:00<?, ?it/s]

                                        composition  yield strength  \
0    (Fe, C, Mn, Si, Cr, Ni, Mo, V, Nb, Co, Al, Ti)          2411.5   
1  (Fe, C, Mn, Si, Cr, Ni, Mo, V, N, Nb, Co, W, Al)          1123.1   
2       (Fe, Mn, Si, Cr, Ni, Mo, V, Nb, Co, Al, Ti)          1736.3   
3    (Fe, C, Mn, Si, Cr, Ni, Mo, V, Nb, Co, Al, Ti)          2487.3   
4    (Fe, C, Mn, Si, Cr, Ni, Mo, V, Nb, Co, Al, Ti)          2249.6   

   MagpieData minimum Number  MagpieData maximum Number  \
0                        6.0                       42.0   
1                        6.0                       74.0   
2                       13.0                       42.0   
3                        6.0                       42.0   
4                        6.0                       42.0   

   MagpieData range Number  MagpieData mean Number  MagpieData avg_dev Number  \
0                     36.0               26.664769                   1.152116   
1                     68.0               26.300744      

Model training and evaluation

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import r2_score, mean_absolute_error

torch.manual_seed(42)

# 1. Define X and y from the featurized dataframe
y = df_features["yield strength"]
X = df_features.drop(columns=["composition", "yield strength"])

# 2. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale input features and target variable
scaler_x = MinMaxScaler()
X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled = scaler_x.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

# 4. Convert arrays to PyTorch tensors (using scaled targets)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# Package training data into batches using DataLoader
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)

# 5. Define the PyTorch MLP Model Architecture
class SteelMLP(nn.Module):
    def __init__(self, input_dim):
        super(SteelMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 1)
        )

    def forward(self, x):
        return self.net(x)

model = SteelMLP(X_train_scaled.shape[1])

# 6. Define Criterion, Optimizer, and Training Configuration (reduced learning rate for stability)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 1000

# 7. Training Loop across Epochs
for epoch in range(epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

# 8. Evaluate Model Performance (inverse transform predictions back to original units)
model.eval()
with torch.no_grad():
    test_preds_scaled = model(X_test_tensor).numpy()
    test_preds = scaler_y.inverse_transform(test_preds_scaled)
    y_test_original = scaler_y.inverse_transform(y_test_tensor.numpy())

print("PyTorch MLP R2 Score:", r2_score(y_test_original, test_preds))
print("PyTorch MLP MAE:", mean_absolute_error(y_test_original, test_preds))



PyTorch MLP R2 Score: 0.7878348231315613
PyTorch MLP MAE: 89.32853698730469
